# Comprobamos que todos los ficheros tienen los mismos campos de información

Nos interesa recuperar la siguiente información: 

- Income Statement:
    * Total Renevues: Total de ingresos de la empresa
- Cash Flow: 
    * % Free Cash Flow Margins: Margen del flujo de caja libre. 
- Ratios: 

    - Return Ratios: Ratios de rentabilidad
        * Return on Assets %: Rentabilidad sobre los activos
        * Return on Invested Capital %: Rentabilidad sobre el capital.
        * Return On Equity %: Rentabilidad sobre los recursos propios.
        * Normalized ROIC %: Rentabilidad sobre las acciones orginarias.

    - Análisis de los márgenes:
        * Gross Profit Margin %: Margen de beneficio bruto.
        * EBITDA Margin %: Margen EBITDA (EBIT + Depreciaciones y Amortizaciones)
        * Net Income Margin %: Margen de beneficio neto
        * Normalized Net Income Margin %: Margen de beneficio neto normalizado
    
    - Short Term Liquidity: Solvencia a corto plazo
        * Current Ratio: Ratio de liquidez (Activos actuales / pasivos actuales)
    
    - Long-Term Liquidity: Solvencia a largo plazo
        * Total Debt / Equity: Deuda total / Fondos propios

In [1]:
import os
os.chdir("C:/TFM/tfm_env/")

import polars as pl
import pandas as pd


In [411]:
# %%writefile src/preprocessdata/utils.py

import pandas as pd
from dataclasses import dataclass
from typing import Any, Callable, Optional, Iterable, Literal
from pandas import DataFrame
import polars as pl
import uuid

# ------------------------------------------------------------
# Tipado estilo steps
# ------------------------------------------------------------

Reader = Callable[[], pd.DataFrame]
Step = Callable[[pd.DataFrame], pd.DataFrame]
Writer = Callable[[pd.DataFrame], None]




def chain(df: pd.DataFrame, *steps: Step) -> pd.DataFrame:
    """
    Aplica los steps en orden usando .pipe para un estilo encadenado.
    """
    for s in steps:
        df = df.pipe(s)
    return df


def pump_load(reader: Reader) -> Callable[[], DataFrame]:
    def _apply() -> DataFrame:
        return reader()
    return _apply


# def func():
#     @requ
#     def _apply(df)-> pd.DataFrame:


# ----------------------------------------------------------------------
# Config
# ----------------------------------------------------------------------

@dataclass(frozen=True)
class PumpConfig:
    base_path: str
    sheet_name: str = None
    sheet_list: list = None
    format: Literal["parquet", "csv", "json", "excel"] = "csv"

@dataclass(frozen=True)
class SinkConfig:
    base_path: str
    mode: Literal["overwrite", "append"] = "overwrite"
    format: Literal["parquet", "csv", "json"] = "csv"


# ----------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------

def _join_path(base: str, name_or_path: str) -> str:
    if name_or_path.startswith(("s3://", "s3a://", "hdfs://", "dbfs:/", "/")):
        return name_or_path
    return f"{base.rstrip('/')}/{name_or_path.lstrip('/')}"


def _select_columns(df: pd.DataFrame, columns: Optional[Iterable[str]]) -> pd.DataFrame:
    if columns is None:
        return df
    cols = list(columns)
    # read_parquet y read_csv soportan proyección; read_json no siempre -> seleccionamos después
    return df.loc[:, [c for c in cols if c in df.columns]]


def _basename(fmt: str) -> str:
    return f"part-{uuid.uuid4().hex}.{fmt}"



def pump_load(loader):
    def _apply():
        return loader()
    return _apply()

# ----------------------------------------------------------------------
# Loader principal (pandas)
# ----------------------------------------------------------------------

def make_loader(
                cfg: PumpConfig,
                name_or_path: str,
                ) -> Reader:
    
    path = _join_path(cfg.base_path, name_or_path)
    fmt = cfg.format.lower()

    def _load() -> pd.DataFrame:
        if fmt == "parquet":
            # pandas puede proyectar columnas en el propio lector
            df = pl.read_parquet(
                path)
            
            df = df.to_pandas()
            return df
        elif fmt == "csv":
            df = pl.read_csv(
                path
                )
            df = df.to_pandas()
            return df
        elif fmt == "json":
            # JSON es variado; pasa flags en context["read_json_kwargs"] (p.ej. lines=True)
            df = pl.read_json(
                path
            )
            df = df.to_pandas()
            df = _select_columns(df, columns)
            return df
        elif fmt == "excel":
            
            if cfg.sheet_name is not None:
                df = pl.read_excel(
                    path,
                    sheet_name=cfg.sheet_name
                )
            else:
                df = pl.read_excel(
                    path
                )
            df = df.to_pandas()
            return df
            
        else:
            raise ValueError(f"Formato no soportado: {fmt}")


    return _load




# ------------------------------------------------------------
# TAPS: efectos controlados (escriben / registran y devuelven DF)
# ------------------------------------------------------------
def tap_write(writer: Writer) -> Step:
    def _apply(df: pd.DataFrame) -> pd.DataFrame:
        writer(df)  # efecto deliberado
        return df
    return _apply




# ------------------------------------------------------------
# Writer principal
# ------------------------------------------------------------
def make_writer(cfg: SinkConfig, name: str, partitionBy: Iterable[str] = ()) -> Writer:
    path = _join_path(cfg.base_path, name)
    fmt = cfg.format.lower()
    parts = list(partitionBy)

    def _writer_csv_or_json(df: pd.DataFrame, ext: str) -> None:
        import fsspec  # ligero; útil para s3/hdfs/file
        fs, _, _ = fsspec.get_fs_token_paths(path)
        base_dir = path
        if cfg.mode == "overwrite" and fs.exists(base_dir):
            fs.rm(base_dir, recursive=True)
        fs.mkdirs(base_dir, exist_ok=True)

        if parts:
            # Escribimos un archivo por partición en subdirectorios estilo col=val/…
            grouped = df.groupby(parts, dropna=False, sort=False)
            for keys, sub in grouped:
                # keys puede ser escalar o tupla
                keys = (keys,) if not isinstance(keys, tuple) else keys
                subdir = "/".join(f"{col}={val}" for col, val in zip(parts, keys))
                full_dir = f"{base_dir.rstrip('/')}/{subdir}"
                fs.mkdirs(full_dir, exist_ok=True)
                file_path = path
                with fs.open(file_path, "wb") as f:
                    if ext == "csv":
                        sub.to_csv(f, index=False)
                    else:
                        # Para JSON, lo más práctico en pipelines: lines=True
                        sub.to_json(f, orient="records", lines=True)
        else:
            # Dataset “sin partición”: escribimos un archivo por llamada (append = archivo nuevo)
            file_path = path
            with fs.open(file_path, "wb") as f:
                if ext == "csv":
                    df.to_csv(f, index=False)
                else:
                    df.to_json(f, orient="records", lines=True)

    def _writer(df: pd.DataFrame) -> None:
        if fmt == "parquet":
            _writer_parquet(df)
        elif fmt == "csv":
            _writer_csv_or_json(df, "csv")
        elif fmt == "json":
            _writer_csv_or_json(df, "json")

        else:
            raise ValueError(f"Formato no soportado: {cfg.format}")

    return _writer

In [406]:
# simple_file_writer.py
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Literal, Optional
import pandas as pd

Writer = Callable[[pd.DataFrame], None]

@dataclass(frozen=True)
class SinkConfig:
    base_path: Path
    format: Literal["parquet", "csv", "json"] = "parquet"
    mode:   Literal["overwrite", "append"] = "overwrite"
    # JSON
    json_lines: bool = True     # True => NDJSON (recomendado para append)
    # CSV
    encoding: str = "utf-8"
    newline: str = ""
    # Parquet
    parquet_engine: str = "pyarrow"  # o "fastparquet"

def _ensure_suffix(path: Path, fmt: str) -> Path:
    ext = { "parquet": ".parquet", "csv": ".csv", "json": ".json" }[fmt]
    return path if path.suffix.lower() == ext else path.with_suffix(ext)

def make_writer(cfg: SinkConfig, name: str) -> Writer:
    """Crea un writer que guarda SIEMPRE en un único archivo local."""
    dest = _ensure_suffix(Path(cfg.base_path + "/" + name), cfg.format)
    dest.parent.mkdir(parents=True, exist_ok=True)

    def _writer(df: pd.DataFrame) -> None:
        if dest.exists() and dest.is_dir():
            raise IsADirectoryError(f"'{dest}' es un directorio; se esperaba un archivo.")

        if cfg.format == "csv":
            if cfg.mode == "overwrite":
                df.to_csv(dest, index=False, encoding=cfg.encoding)
            else:  # append
                header = not dest.exists()
                df.to_csv(dest, mode="a", header=header, index=False,
                          encoding=cfg.encoding)

        elif cfg.format == "json":
            if cfg.json_lines:  # NDJSON: perfecto para append
                mode = "w" if cfg.mode == "overwrite" else "a"
                with open(dest, mode, encoding=cfg.encoding, newline=cfg.newline) as f:
                    df.to_json(f, orient="records", lines=True, force_ascii=False)
            else:
                # JSON como array. Append simple = leer + concatenar + reescribir.
                if cfg.mode == "append" and dest.exists():
                    import json
                    with open(dest, "r", encoding=cfg.encoding) as f:
                        existing = json.load(f)
                    if not isinstance(existing, list):
                        raise ValueError("El JSON existente no es un array; no puedo hacer append.")
                    existing.extend(df.to_dict(orient="records"))
                    with open(dest, "w", encoding=cfg.encoding, newline=cfg.newline) as f:
                        json.dump(existing, f, ensure_ascii=False)
                else:
                    df.to_json(dest, orient="records", force_ascii=False)

        elif cfg.format == "parquet":
            if cfg.mode == "append" and dest.exists():
                # Estrategia simple: leer existente, concatenar y reescribir.
                prev = pd.read_parquet(dest, engine=cfg.parquet_engine)
                out = pd.concat([prev, df], ignore_index=True)
                out.to_parquet(dest, index=False, engine=cfg.parquet_engine)
            else:
                df.to_parquet(dest, index=False, engine=cfg.parquet_engine)

        else:
            raise ValueError(f"Formato no soportado: {cfg.format}")

    return _writer


In [385]:
base_dir = "C:/TFM/tfm_env/data/01_raw/prueba/"
ext="csv"
f"{base_dir.rstrip('/')}"

'C:/TFM/tfm_env/data/01_raw/prueba'

In [ ]:
# etl/sink_pd.py
from __future__ import annotations
from dataclasses import dataclass
from typing import Any, Callable, Dict, Iterable, Optional, Literal

import pandas as pd


def tap_metrics(log: Callable[[Dict[str, Any]], None]) -> Step:
    def _apply(df: pd.DataFrame) -> pd.DataFrame:
        rows = len(df)
        distinct_id = df["id"].nunique(dropna=True) if "id" in df.columns else None
        log({"rows": rows, "distinct_id": distinct_id})
        return df
    return _apply

# ------------------------------------------------------------
# Config del sink y utilidades
# ------------------------------------------------------------



In [ ]:
# etl/loader_pd.py
from __future__ import annotations
from dataclasses import dataclass
from typing import Any, Callable, Optional, Iterable, Literal

import pandas as pd

Reader = Callable[[], pd.DataFrame]

# ----------------------------------------------------------------------
# Config
# ----------------------------------------------------------------------

@dataclass(frozen=True)
class PumpConfig:
    base_path: str
    context: Any = None  # p.ej. dict con storage_options/kwargs
    format: Literal["parquet", "csv", "json"] = "csv"

# ----------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------

def _join_path(base: str, name_or_path: str) -> str:
    if name_or_path.startswith(("s3://", "s3a://", "hdfs://", "dbfs:/", "/")):
        return name_or_path
    return f"{base.rstrip('/')}/{name_or_path.lstrip('/')}"


def _select_columns(df: pd.DataFrame, columns: Optional[Iterable[str]]) -> pd.DataFrame:
    if columns is None:
        return df
    cols = list(columns)
    # read_parquet y read_csv soportan proyección; read_json no siempre -> seleccionamos después
    return df.loc[:, [c for c in cols if c in df.columns]]

# ----------------------------------------------------------------------
# Loader principal (pandas)
# ----------------------------------------------------------------------

def make_loader(
                cfg: PumpConfig,
                name_or_path: str,
                *,
                columns: Optional[Iterable[str]] = None,
                alias: Optional[str] = None,
                ) -> Reader:
    
    path = _join_path(cfg.base_path, name_or_path)
    fmt = cfg.format.lower()

    def _load() -> pd.DataFrame:
        if fmt == "parquet":
            # pandas puede proyectar columnas en el propio lector
            df = pd.read_parquet(
                path,
                columns=list(columns) if columns else None,)
        elif fmt == "csv":
            df = pd.read_csv(
                path,
                usecols=list(columns) if columns else None,
                s
            )
        elif fmt == "json":
            # JSON es variado; pasa flags en context["read_json_kwargs"] (p.ej. lines=True)
            df = pd.read_json(
                path
            )
            df = _select_columns(df, columns)
        else:
            raise ValueError(f"Formato no soportado: {fmt}")

        # “Alias” no existe en pandas; lo guardamos en metadatos del DF
        if alias:
            try:
                df.attrs["alias"] = alias
            except Exception:
                pass
        return df

    return _load


In [2]:
from src.financial_metrics_check import *

In [434]:


def is_ratio_sheet(file):
    if file[:-4].endswith("Ratios"):
        return True
    return False

def is_cash_flow_sheet(file):
    if file[:-4].endswith("Cash Flow"):
        return True
    return False

def check_field_in_ratio_sheet(df, field):

    field_list = df['Ratios | TIKR.com'].apply(lambda x: x.lower()).tolist()
    return field.lower() in field_list


def is_valid_file_ratio(file_path, ratio_list):

    for field in ratio_list:
        if not check_field_in_ratio_sheet(file_path, field):
            return False
    return True

def check_field_in_cash_flow_sheet(df, field):

    field_list = df['Cash Flow | TIKR.com'].apply(lambda x: x.lower()).tolist()
    return field.lower() in field_list


def is_valid_file_cash_flow(file_path, cash_flow_list):

    for field in cash_flow_list:
        if not check_field_in_cash_flow_sheet(file_path, field):
            return False
    return True


def check_field_in_income_statement(df, field):

    field_list = df['Income Statement | TIKR.com'].apply(lambda x: x.lower()).tolist()
    return field.lower() in field_list


def is_valid_file_income_statement(file_path, income_statement_list):

    for field in income_statement_list:
        if not check_field_in_income_statement(file_path, field):
            return False
    return True



def extract_tikr(file_name='file_name.xlsx'):
    
    return file_name.split("-")[1].strip()
    
    
def extract_currency(file_name='file_name.xlsx'):
    
    return file_name.split("-")[0].strip()
    
    
ratios_list = ["Return on Assets %", "Return on Invested Capital %", 
               "Return On Equity %", "Normalized ROIC %", "Gross Profit Margin %", 
               "EBITDA Margin %", "Net Income Margin %", 
               "Normalized Net Income Margin %", "Current Ratio", 
               "Total Debt / Equity"]



In [431]:
from pathlib import Path
Path("C:/TFM/tfm_env/data/01_raw/").joinpath("prueba/").as_posix()

'C:/TFM/tfm_env/data/01_raw/prueba'

In [ ]:
pump_raw_historical_financial_statement_ratios = PumpConfig("C:/TFM/tfm_env/data/01_raw/01.3 - HistoricoEstadosFinancieros/", format="excel", sheet_name="Ratios")
pump_raw_historical_financial_statement = PumpConfig("C:/TFM/tfm_env/data/01_raw/01.3 - HistoricoEstadosFinancieros/", format="excel", sheet_list=["Ratios", "Cash Flow"])



raw_sink = SinkConfig(base_path="data/01_raw/", format="csv")

def valid_files_gen(cfg: PumpConfig) -> str:
    path = Path(cfg.base_path)

    for sector in [s for s in os.listdir(path)]: 
        sector_path = path.joinpath(sector)
        for file in os.listdir(sector_path):
            print(file)

            file_list = []
            
            for sheet in cfg.sheet_list:

                file_info = {
                    "base_path": path,
                    "sector": sector,
                    "sheet": sheet,
                    "file": file
                }
                
                df = pl.read_excel(f"{path}/{sector}/{file}".format_map(file_info), sheet = file_info["sheet"])
                
                
                
                
                
                # chain(
#     pump_load(make_loader(
#                 cfg= pump_raw_historical_financial_statement,
#                 name_or_path= "{sector_dir}{file}".format_map(file_info)
#                 )),
    
#     tap_write(make_writer(cfg=raw_sink, name="prueba/prueba2"))
# )



gen = valid_files_gen(pump_raw_historical_financial_statement)


# def step_fun() -> DataFrame:
#     def _apply(df):
#         return df
#     return _apply
from datetime import datetime


# def step_check_valid_historical_period(sheet_name) -> DataFrame:
#     def _apply(df):
        
#         title_idx_cash_flow = sheet_name + " | TIKR.com"

#         l = df.keys().tolist()
#         l.remove(sheet_name + " | TIKR.com")
#         l.remove("LTM")
#         l = [datetime.strptime(d, "%d/%m/%y") for d in l]
#         if (max(l).year<2024) or (len(l)<16):
#             continue
        

        
#         return df
#     return _apply






# # # Eslabón más bajo
# chain(
#     pump_load(make_loader(
#                 cfg= pump_raw_historical_financial_statement,
#                 name_or_path= "{sector_dir}{file}".format_map(file_info)
#                 )),
    
#     tap_write(make_writer(cfg=raw_sink, name="prueba/prueba2"))
# )


AUD - CHC - Financials (30.6.07 - 30.6.24).xlsx
AUD - GPT - Financials (31.12.04 - 31.12.24).xlsx
AUD - MGR - Financials (30.6.03 - 30.6.24).xlsx
AUD - SGP - Financials (30.6.04 - 30.6.24).xlsx
CAD - HR.UN - Financials (31.12.05 - 31.12.24).xlsx
EUR - COV - Financials (31.12.05 - 31.12.24).xlsx
EUR - ICAD - Financials (31.12.05 - 31.12.24).xlsx
EUR - MRL - Financials (31.12.13 - 31.12.24).xlsx
GBP - BLND - Financials (31.3.06 - 31.3.25).xlsx
GBP - LAND - Financials (31.3.06 - 31.3.25).xlsx
JPY - 3279 - Financials (30.11.12 - 30.11.24).xlsx
JPY - 3309 - Financials (31.10.15 - 31.10.24).xlsx
JPY - 3462 - Financials (31.8.14 - 31.8.24).xlsx
JPY - 8960 - Financials (30.11.06 - 30.11.24).xlsx
JPY - 8972 - Financials (31.10.06 - 31.10.24).xlsx
JPY - 8984 - Financials (31.8.09 - 31.8.24).xlsx
MXN - DANHOS 13 - Financials (31.12.13 - 31.12.24).xlsx
MXN - FUNO 11 - Financials (31.12.11 - 31.12.24).xlsx
MYR - KLCC - Financials (31.3.05 - 31.12.24).xlsx
SGD - N2IU - Financials (31.3.11 - 31.3.25)

USD - WFRD - Financials (31.12.05 - 31.12.24).xlsx
ZAR - EXX - Financials (31.12.05 - 31.12.24).xlsx
AED - ADCB - Financials (31.12.05 - 31.12.24).xlsx
AED - DFM - Financials (31.12.05 - 31.12.24).xlsx
AED - DIB - Financials (31.12.05 - 31.12.24).xlsx
AED - EMIRATESNBD - Financials (31.12.06 - 31.12.24).xlsx
AED - RAKBANK - Financials (31.12.06 - 31.12.24).xlsx
ARS - GGAL - Financials (31.12.05 - 31.12.24).xlsx
AUD - AFI - Financials (30.6.05 - 30.6.24).xlsx
AUD - ARG - Financials (30.6.02 - 30.6.24).xlsx
AUD - BOQ - Financials (31.8.05 - 31.8.24).xlsx
AUD - BPT - Financials (30.6.06 - 30.6.25).xlsx
AUD - MQG - Financials (31.3.03 - 31.3.25).xlsx
AUD - PNI - Financials (30.6.07 - 30.6.25).xlsx
AUD - QBE - Financials (31.12.02 - 31.12.24).xlsx
AUD - WBC - Financials (30.9.05 - 30.9.24).xlsx
BRL - BBAS3 - Financials (31.12.05 - 31.12.24).xlsx
BRL - ITSA4 - Financials (31.12.05 - 31.12.24).xlsx
BRL - PSSA3 - Financials (31.12.05 - 31.12.24).xlsx
CAD - BNS - Financials (31.10.05 - 31.10.24

In [ ]:
# parallel_utils.py
from __future__ import annotations
from typing import Iterable, Callable, TypeVar, Generic, Any, Union, List
from joblib import Parallel, delayed

T = TypeVar("T")
R = TypeVar("R")

def parallel_map(
    items: Iterable[T],
    func: Callable[[T], R],
    *,
    n_jobs: int = -1,                 # -1 = todos los cores
    prefer: str = "threads",          # "threads" (I/O) | "processes" (CPU)
    batch_size: Union[int, str, None] = "auto",
    progress: bool = True,
    on_error: str = "raise",          # "raise" | "return"
) -> List[R]:
    """
    Aplica func(item) en paralelo sobre 'items' usando joblib.
    - on_error="raise": la primera excepción detiene todo.
      on_error="return": devuelve la excepción en el lugar del resultado.
    - progress=True: intenta mostrar barra de progreso con tqdm (si está instalado).
    """
    it = list(items)
    if progress:
        try:
            from tqdm.auto import tqdm  # opcional
            it = tqdm(it, desc="Processing", unit="task")
        except Exception:
            pass

    def _safe(item: T) -> R:
        try:
            return func(item)
        except Exception as e:               # type: ignore[return-value]
            if on_error == "raise":
                raise
            return e                         # devuelve la excepción como resultado

    results: List[R] = Parallel(n_jobs=n_jobs, prefer=prefer, batch_size=batch_size)(
        delayed(_safe)(x) for x in it
    )
    return results


In [ ]:
import json

with open("C:/TFM/tfm_env/data/01_raw/01.3.2 AllTickersInfo/" + "AllTickersInfo.json", "r") as file:
    all_ticker_info = json.load(file)

inv_all_ticker_info = {sector: {v: k for k, v in all_ticker_info[sector].items()} for sector in all_ticker_info.keys()}

In [ ]:
import logging

# Configuración básica del logger
logging.basicConfig(
    filename="C:/TFM/tfm_env/data/02_intermediate/02.2.3 SeleccionHistoricoEstadosFinancierosUnificadosTodosLosSectores/logs/app.log",              # Nombre del archivo de logs
    level=logging.INFO,              # Nivel mínimo (DEBUG, INFO, WARNING, ERROR, CRITICAL)
    format="%(asctime)s - %(levelname)s - %(message)s"  # Formato del log
)

# # Ejemplo de uso
# logging.debug("Esto es un mensaje DEBUG (no se guardará porque el nivel está en INFO).")
# logging.info("Inicio de la aplicación")
# logging.warning("Cuidado, algo no va del todo bien...")
# logging.error("Ha ocurrido un error")
# logging.critical("Error crítico, se debe detener el programa")


In [456]:
from datetime import datetime

logging.info("Inicio del programa.\n")

statement_data_dir = "data/01_raw/01.3 - HistoricoEstadosFinancieros/"
statement_data_dir_dest = "data/02_intermediate/02.1 - HistoricoEstadosFinancieros/"
selected_data_dir_dest = "data/02_intermediate/02.2 - SeleccionHistoricoEstadosFinancieros/"
sheets = ["Income Statement", "Ratios", "Cash Flow"]

sectors = [dir + "/" for dir in os.listdir(statement_data_dir)]

dataframe = pd.DataFrame()

dataframe["Sector"] = []
dataframe["% Free Cash Flow Margins"] = []




for r in ratios_list:

    dataframe[r] = []

h = True
for sector in sectors[:-1]: # Al final, eliminar los corchetes y dejar como lista. Ahora fijamos solo ConsumoDiscrecional
    for file in os.listdir(statement_data_dir + sector):
        # logging.info(f"Entrando en ", statement_data_dir + sectors[1])
        
        valid_cash_flow_file = False
        valid_ratio_file = False
        valid_income_statement_file = False
        for sheet in sheets:
            # logging.info(f"Leyendo hoja: {sheet}")
            
            df = pl.read_excel(f"{statement_data_dir}{sector}{file}", sheet_name=sheet)


            if sheet == "Income Statement":
                income_statement_dataframe = df.to_pandas()
                title_idx_income_statement = sheet + " | TIKR.com"

                l = income_statement_dataframe.keys().tolist()
                l.remove(sheet + " | TIKR.com")
                l.remove("LTM")
                l = [datetime.strptime(d, "%d/%m/%y") for d in l]
                if (max(l).year<2024) or (len(l)<16):
                    continue
                
                valid_income_statement_file = is_valid_file_income_statement(income_statement_dataframe, ["Total Revenues"])
                


            if sheet == "Cash Flow":
                cash_flow_dataframe = df.to_pandas()
                title_idx_cash_flow = sheet + " | TIKR.com"

                l = cash_flow_dataframe.keys().tolist()
                l.remove(sheet + " | TIKR.com")
                l.remove("LTM")
                l = [datetime.strptime(d, "%d/%m/%y") for d in l]
                if (max(l).year<2024) or (len(l)<16):
                    continue
                
                valid_cash_flow_file = is_valid_file_cash_flow(cash_flow_dataframe, ["% Free Cash Flow Margins"])
                
            if sheet == "Ratios":
                ratios_dataframe = df.to_pandas()
                title_idx_ratio = sheet + " | TIKR.com"
                l = ratios_dataframe.keys().tolist()
                l.remove(title_idx_ratio)
                l.remove("LTM")
                l = [datetime.strptime(d, "%d/%m/%y") for d in l]
                if (max(l).year<2024) or (len(l)<16):
                    continue
                
                valid_ratio_file = is_valid_file_ratio(ratios_dataframe, ratios_list)


        if (valid_cash_flow_file and valid_ratio_file) and valid_income_statement_file:
            
            
            i = income_statement_dataframe.set_index(title_idx_income_statement)
            i_cols=i.keys().tolist()
            i_cols.remove("LTM")
            i_serie = i[i_cols].transpose()["Total Revenues"]
            
            
            c = cash_flow_dataframe.set_index(title_idx_cash_flow, )

            c_cols = c.keys().tolist()
            c_cols.remove("LTM")
            fcf_serie = c[c_cols].transpose()["% Free Cash Flow Margins"]


            r = ratios_dataframe.set_index(title_idx_ratio)

            r_cols = r.keys().tolist()
            r_cols.remove("LTM")

            r_df = r[r_cols].transpose()[ratios_list]

            data = pd.concat([r_df, fcf_serie, i_serie], axis=1)
            
            
            ticker_name = extract_tikr(file)
            statement_currency = extract_currency(file)
            data["statement currency"] = statement_currency
            data["ticker"] = ticker_name
            data["sector"] = sector[:-1]
            data.reset_index(inplace=True)
            data = data.rename(columns={"index": "Date"})
            
            
            ticker = all_ticker_info[sector[:-1]][ticker_name]
            historical_price_company_dir="data/01_raw/01.4 - DatosHistoricosPrecios/Empresas/"
            hist_df =None
            
            for file_historical_price in os.listdir(historical_price_company_dir):
                if file_historical_price.split("_")[0] == ticker:
                    hist_df = pd.read_csv(f"{historical_price_company_dir}{file_historical_price}", header=[0, 1, 2])
                    close_currency = file_historical_price.split("_")[1].replace(".csv", "")
                    break
            
            if hist_df is None:
                continue
            
            data["Date"] = data.Date.apply(lambda x: pd.to_datetime(x, utc=True))
            data["year"] = data.Date.apply(lambda x: x.year)

            C = hist_df["Close"].transpose().reset_index().transpose().iloc[2:]
            C.rename(columns={0: "Close"}, inplace=True)
            C["Date"] = hist_df["Price"]["Ticker"]["Date"]
            C["Date"] = C.Date.apply(lambda x: pd.to_datetime(x, utc=True))
            C["year"] = C.Date.apply(lambda x: x.year)
            C["close currency"] = close_currency
            C = C.set_index("Date").join(data[data.ticker==ticker_name].set_index("year"), on="year", how="left", lsuffix="l_", rsuffix="r_")
            
            data.set_index("Date")
            
            
            
            
            data.to_csv(f"{selected_data_dir_dest}{ticker_name}-data.csv", index_label="Date")
            data.to_csv(f"data/02_intermediate/02.2.3 SeleccionHistoricoEstadosFinancierosUnificadosTodosLosSectores/AllData.csv", mode="a", index_label="Date", header=h)
            C.to_csv(f"data/02_intermediate/02.2.3 SeleccionHistoricoEstadosFinancierosUnificadosTodosLosSectores/AllDataClosePrices.csv", mode="a", index_label="Date", header=h)
            h = False
            
            for sheet in sheets:

                df = pl.read_excel(f"{statement_data_dir}{sector}{file}", sheet_name=sheet)
                nombre_csv = f"{statement_data_dir_dest}{sector}{file[:-5]}-{sheet}.csv"
                # logging.info(nombre_csv)
                df.write_csv(nombre_csv)
                logging.info(f"Guardado: {nombre_csv}")
                
            

logging.info("El script finalizó.\n")


In [451]:
inv_all_ticker_info

{'BienesRaíces': {'CHC.AX': 'CHC',
  'GPT.AX': 'GPT',
  'MGR.AX': 'MGR',
  'SGP.AX': 'SGP',
  'HR-UN.TO': 'HR.UN',
  'COV.PA': 'COV',
  'ICAD.PA': 'ICAD',
  'MRL.MC': 'MRL',
  'BLND.L': 'BLND',
  'LAND.L': 'LAND',
  '3279.T': '3279',
  '3309.T': '3309',
  '3462.T': '3462',
  '8960.T': '8960',
  '8972.T': '8972',
  '8984.T': '8984',
  'DANHOS13.MX': 'DANHOS 13',
  'FUNO11.MX': 'FUNO 11',
  'KLCC.KL': 'KLCC',
  'N2IU.SI': 'N2IU',
  'T82U.SI': 'T82U',
  'ZRGYO.IS': 'ZRGYO',
  'BNL': 'BNL',
  'EPRT': 'EPRT',
  'ESBA': 'ESBA',
  'LMT': 'LMT',
  'WPC': 'WPC',
  'GRT.JO': 'GRT'},
 'ConsumoDiscrecional': {'TALABAT.AE': 'TALABAT',
  'BRG.AX': 'BRG',
  'PMV.AX': 'PMV',
  'SMFT3.SA': 'SMFT3',
  'VBBR3.SA': 'VBBR3',
  'DOL.TO': 'DOL',
  'HBM.TO': 'HBM',
  'MG.TO': 'MG',
  'AVOL.SW': 'AVOL',
  'UHR.SW': 'UHR',
  'COPEC.SN': 'COPEC',
  '000559.SZ': '000559',
  '000564.SZ': '000564',
  '000625.SZ': '000625',
  '000887.SZ': '000887',
  '002031.SZ': '002031',
  '002032.SZ': '002032',
  '002085.SZ': '00

In [433]:
data

,Date,Return on Assets %,Return on Invested Capital %,Return On Equity %,Normalized ROIC %,Gross Profit Margin %,EBITDA Margin %,Net Income Margin %,Normalized Net Income Margin %,Current Ratio,Total Debt / Equity,% Free Cash Flow Margins,statement currency,ticker,sector,year
0,2005-06-30 00:00:00+00:00,0.119474,0.228549,0.322841,0.229128,,,,,"2,75x",1.085531,-5.505000e+08,ZAR,WHL,ConsumoDiscrecional,2005
1,2006-06-30 00:00:00+00:00,0.105058,0.211413,0.348945,0.211801,,,,,"1,88x",0.814459,-6.673000e+08,ZAR,WHL,ConsumoDiscrecional,2006
2,2007-06-30 00:00:00+00:00,0.111854,0.196765,0.368874,0.198564,0.343962,0.127572,0.061829,0.062537,"1,76x",1.22021,-4.246442e-02,ZAR,WHL,ConsumoDiscrecional,2007
3,2008-06-30 00:00:00+00:00,0.086335,0.190569,0.276615,0.191861,0.312317,0.099687,0.046684,0.047107,"0,94x",0.687608,-9.917817e-03,ZAR,WHL,ConsumoDiscrecional,2008
4,2009-06-30 00:00:00+00:00,0.127596,0.250081,0.381028,0.276762,0.338518,0.113493,0.056929,0.057477,"1,68x",0.511405,1.126722e-02,ZAR,WHL,ConsumoDiscrecional,2009
5,2010-06-30 00:00:00+00:00,0.145308,0.260569,0.391839,0.289163,0.332322,0.100543,0.053777,0.054204,"1,28x",0.457892,3.355705e-02,ZAR,WHL,ConsumoDiscrecional,2010
6,2011-06-30 00:00:00+00:00,0.18047,0.322293,0.440573,0.3401,0.347862,0.093425,0.063756,0.063756,"1,41x",0.245758,3.228833e-02,ZAR,WHL,ConsumoDiscrecional,2011
7,2012-06-30 00:00:00+00:00,0.210097,0.379487,0.473833,0.396759,0.356069,0.122815,0.07027,0.071598,"1,18x",0.221626,3.083485e-02,ZAR,WHL,ConsumoDiscrecional,2012
8,2013-06-30 00:00:00+00:00,0.22764,0.409914,0.500592,0.425462,0.384733,0.126891,0.071962,0.073722,"1,23x",0.233369,2.077952e-02,ZAR,WHL,ConsumoDiscrecional,2013
9,2014-06-30 00:00:00+00:00,0.16158,0.245462,0.453546,0.261091,0.390309,0.128743,0.070139,0.072733,"1,05x",1.400211,1.372554e-02,ZAR,WHL,ConsumoDiscrecional,2014


In [ ]:
# import json
# dest_path = "C:/TFM/tfm_env/data/02_intermediate/02.2.2 EmparejamientoTickerExchange/"

# # Guardar en archivo JSON
# with open(dest_path + "tiker_consumo_discrecional2.json", "w", encoding="utf-8") as f:
#     json.dump(new_discretional_tickers, f, indent=4, ensure_ascii=False)



In [153]:
data.head()

,Return on Assets %,Return on Invested Capital %,Return On Equity %,Normalized ROIC %,Gross Profit Margin %,EBITDA Margin %,Net Income Margin %,Normalized Net Income Margin %,Current Ratio,Total Debt / Equity,% Free Cash Flow Margins,ticker,sector
30/6/05,0.119474,0.228549,0.322841,0.229128,,,,,"2,75x",1.085531,-5.505000e+08,WHL,ConsumoDiscrecional
30/6/06,0.105058,0.211413,0.348945,0.211801,,,,,"1,88x",0.814459,-6.673000e+08,WHL,ConsumoDiscrecional
30/6/07,0.111854,0.196765,0.368874,0.198564,0.343962,0.127572,0.061829,0.062537,"1,76x",1.22021,-4.246442e-02,WHL,ConsumoDiscrecional
30/6/08,0.086335,0.190569,0.276615,0.191861,0.312317,0.099687,0.046684,0.047107,"0,94x",0.687608,-9.917817e-03,WHL,ConsumoDiscrecional
30/6/09,0.127596,0.250081,0.381028,0.276762,0.338518,0.113493,0.056929,0.057477,"1,68x",0.511405,1.126722e-02,WHL,ConsumoDiscrecional


In [ ]:
selected_data_dir_dest

for file in os.listdir(selected_data_dir_dest):
    d = pd.read_csv(selected_data_dir_dest + file, index_col="id")
    break

d


,Return on Assets %,Return on Invested Capital %,Return On Equity %,Normalized ROIC %,Gross Profit Margin %,EBITDA Margin %,Net Income Margin %,Normalized Net Income Margin %,Current Ratio,Total Debt / Equity,% Free Cash Flow Margins,ticker,sector
id,,,,,,,,,,,,,
31/12/05,0.027828,0.055699,0.081928,0.058309,0.182241,0.173516,0.046169,0.046169,"1,21x",1.073635,-0.052447,559,ConsumoDiscrecional
31/12/06,0.035960,0.069599,0.108246,0.077537,0.192040,0.166307,0.050959,0.055599,"0,95x",0.851650,0.040267,559,ConsumoDiscrecional
31/12/07,0.049382,0.086620,0.146612,0.091243,0.203532,0.167756,0.062186,0.065505,"0,91x",0.542055,0.017622,559,ConsumoDiscrecional
31/12/08,0.039087,0.066865,0.133089,0.073904,0.169688,0.131086,0.045544,0.050338,"0,78x",1.551826,-0.002920,559,ConsumoDiscrecional
31/12/09,0.045809,0.079689,0.171058,0.092497,0.181175,0.129816,0.047772,0.055450,"0,81x",0.781977,0.099250,559,ConsumoDiscrecional
31/12/10,0.061320,0.131180,0.157621,0.148380,0.176660,0.116431,0.054453,0.060367,"1,35x",0.215893,0.012263,559,ConsumoDiscrecional
31/12/11,0.055250,0.112830,0.123628,0.127275,0.178075,0.115621,0.057743,0.062475,"1,27x",0.164551,-0.028843,559,ConsumoDiscrecional
31/12/12,0.049255,0.114153,0.113270,0.131144,0.172927,0.127724,0.052273,0.056596,"1,75x",0.461065,0.036181,559,ConsumoDiscrecional
31/12/13,0.055343,0.112539,0.139677,0.132235,0.197785,0.131003,0.057289,0.064612,"1,38x",0.617321,0.101476,559,ConsumoDiscrecional


In [437]:
d = pd.read_csv("data/02_intermediate/02.2.1 SeleccionHistoricoEstadosFinancierosUnificado/AllDataClosePrices.csv", index_col="Date")
# d.index = pd.to_datetime(d.index)
# d[d.index.year == 2024]

C:\Users\Edelmín\AppData\Local\Temp\ipykernel_18664\2792004605.py:1: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  d = pd.read_csv("data/02_intermediate/02.2.1 SeleccionHistoricoEstadosFinancierosUnificado/AllDataClosePrices.csv", index_col="Date")


In [438]:
d[d.ticker.notna()]

,Close,year,close currency,Date.1,Return on Assets %,Return on Invested Capital %,Return On Equity %,Normalized ROIC %,Gross Profit Margin %,EBITDA Margin %,Net Income Margin %,Normalized Net Income Margin %,Current Ratio,Total Debt / Equity,% Free Cash Flow Margins,Total Revenues,statement currency,ticker,sector
Date,,,,,,,,,,,,,,,,,,,
2004-01-01 00:00:00+00:00,1.150652,2004,AUD,2004-06-30 00:00:00+00:00,NaN,0.013959,NaN,0.015116,0.408122,0.097632,NaN,NaN,"3,06x",0.443614,0.029603,457.616,AUD,BRG,ConsumoDiscrecional
2004-01-02 00:00:00+00:00,1.141868,2004,AUD,2004-06-30 00:00:00+00:00,NaN,0.013959,NaN,0.015116,0.408122,0.097632,NaN,NaN,"3,06x",0.443614,0.029603,457.616,AUD,BRG,ConsumoDiscrecional
2004-01-05 00:00:00+00:00,1.124301,2004,AUD,2004-06-30 00:00:00+00:00,NaN,0.013959,NaN,0.015116,0.408122,0.097632,NaN,NaN,"3,06x",0.443614,0.029603,457.616,AUD,BRG,ConsumoDiscrecional
2004-01-06 00:00:00+00:00,1.141868,2004,AUD,2004-06-30 00:00:00+00:00,NaN,0.013959,NaN,0.015116,0.408122,0.097632,NaN,NaN,"3,06x",0.443614,0.029603,457.616,AUD,BRG,ConsumoDiscrecional
2004-01-07 00:00:00+00:00,1.124301,2004,AUD,2004-06-30 00:00:00+00:00,NaN,0.013959,NaN,0.015116,0.408122,0.097632,NaN,NaN,"3,06x",0.443614,0.029603,457.616,AUD,BRG,ConsumoDiscrecional
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-23 00:00:00+00:00,6208.781250,2024,ZAc,2024-06-30 00:00:00+00:00,0.066548,0.127521,0.226909,0.140517,0.358917,0.112474,0.033881,0.037267,"0,92x",1.667526,0.001019,76533.000,ZAR,WHL,ConsumoDiscrecional
2024-12-24 00:00:00+00:00,6280.767090,2024,ZAc,2024-06-30 00:00:00+00:00,0.066548,0.127521,0.226909,0.140517,0.358917,0.112474,0.033881,0.037267,"0,92x",1.667526,0.001019,76533.000,ZAR,WHL,ConsumoDiscrecional
2024-12-27 00:00:00+00:00,6201.782715,2024,ZAc,2024-06-30 00:00:00+00:00,0.066548,0.127521,0.226909,0.140517,0.358917,0.112474,0.033881,0.037267,"0,92x",1.667526,0.001019,76533.000,ZAR,WHL,ConsumoDiscrecional


In [294]:
p = "data/01_raw/01.4 - DatosHistoricosPrecios/Empresas"

len(os.listdir(p))

226

In [292]:
d[d.ticker.notna()].ticker.nunique()

47

In [167]:
d["Current Ratio"] = d["Current Ratio"].apply(lambda x: str(x).replace("x", ""))

In [118]:
d.keys()

Index(['Return on Assets %', 'Return on Invested Capital %',
       'Return On Equity %', 'Normalized ROIC %', 'Gross Profit Margin %',
       'EBITDA Margin %', 'Net Income Margin %',
       'Normalized Net Income Margin %', 'Current Ratio',
       'Total Debt / Equity', '% Free Cash Flow Margins', 'ticker', 'sector'],
      dtype='object')

In [168]:
from sklearn.linear_model import LinearRegression
d1 = d.reset_index()

def to_ordinal(s):
    return pd.to_datetime(s).map(pd.Timestamp.toordinal).values.reshape(-1, 1)

def time_regress_impute_group(g: pd.DataFrame, cols, date_col="date"):
    g = g.copy()

    # 1) Asegurar fechas válidas
    g[date_col] = pd.to_datetime(g[date_col], errors="coerce")
    if g[date_col].isna().all():
        return g  # sin fechas válidas no se puede ajustar
    X_all = g[date_col].map(pd.Timestamp.toordinal).to_numpy().reshape(-1, 1)

    for target in cols:
        # 2) Separar train / pred por disponibilidad del target
        mask_train = g[target].notna().to_numpy()
        mask_pred  = ~mask_train

        # Si no hay nada que imputar, saltar
        if mask_pred.sum() == 0:
            continue

        # 3) Filtrar filas de train que tengan X válido y target válido
        X_train = X_all[mask_train]
        y_train = g.loc[mask_train, target].to_numpy()

        # Necesitamos al menos 2 puntos para ajustar una recta
        if len(X_train) < 2:
            continue

        # 4) Ajustar y predecir solo si hay muestras a predecir
        model = LinearRegression()
        model.fit(X_train, y_train)

        X_pred = X_all[mask_pred]
        y_hat  = model.predict(X_pred)

        # 5) Escribir predicciones en las posiciones faltantes
        g.loc[g.index[mask_pred], target] = y_hat

    return g


columns = ['Return on Assets %', 'Return on Invested Capital %',
       'Return On Equity %', 'Normalized ROIC %', 'Gross Profit Margin %',
       'EBITDA Margin %', 'Net Income Margin %',
       'Normalized Net Income Margin %', 'Current Ratio',
       'Total Debt / Equity', '% Free Cash Flow Margins']


df = d1.groupby("ticker", group_keys=False).apply(time_regress_impute_group, cols=columns, date_col="id")


C:\Users\Edelmín\AppData\Local\Temp\ipykernel_18664\3076607675.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  g[date_col] = pd.to_datetime(g[date_col], errors="coerce")
C:\Users\Edelmín\AppData\Local\Temp\ipykernel_18664\3076607675.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  g[date_col] = pd.to_datetime(g[date_col], errors="coerce")
C:\Users\Edelmín\AppData\Local\Temp\ipykernel_18664\3076607675.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  g[date_col] = pd.to_datetime(g[date_col], errors="coerce")
C:\Users\Edelmín\AppData\Local\Temp\ipykernel_18664\307

In [169]:
df.isna().sum()

id                                0
Return on Assets %                0
Return on Invested Capital %      0
Return On Equity %                0
Normalized ROIC %                 0
Gross Profit Margin %             0
EBITDA Margin %                   0
Net Income Margin %               0
Normalized Net Income Margin %    0
Current Ratio                     0
Total Debt / Equity               0
% Free Cash Flow Margins          0
ticker                            0
sector                            0
dtype: int64

In [172]:
df

,id,Return on Assets %,Return on Invested Capital %,Return On Equity %,Normalized ROIC %,Gross Profit Margin %,EBITDA Margin %,Net Income Margin %,Normalized Net Income Margin %,Current Ratio,Total Debt / Equity,% Free Cash Flow Margins,ticker,sector
0,2004-06-30,0.052282,0.013959,0.074506,0.015116,0.408122,0.097632,0.024784,0.052338,"3,06",0.443614,0.029603,BRG,ConsumoDiscrecional
1,2005-06-30,0.046868,0.071712,0.083124,0.072656,0.421123,0.064129,0.029999,0.029999,"3,26",0.495526,-0.000862,BRG,ConsumoDiscrecional
2,2007-06-30,-0.123240,-0.143022,-0.290219,0.099171,0.359705,0.082380,-0.091546,0.045792,"1,56",0.806993,0.010370,BRG,ConsumoDiscrecional
3,2008-06-30,0.079251,0.126275,0.167958,0.129324,0.340271,0.089465,0.049098,0.049328,"1,93",0.407044,0.054535,BRG,ConsumoDiscrecional
4,2009-06-30,0.048150,0.089752,0.089392,0.091143,0.350291,0.065083,0.027214,0.027214,"2,13",0.299270,0.046564,BRG,ConsumoDiscrecional
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3116,2020-06-30,0.011030,0.062931,0.069988,0.078870,0.351055,0.122729,0.007714,0.015681,"1,01",7.045616,0.053748,WHL,ConsumoDiscrecional
3117,2021-06-30,0.068974,0.128280,0.526909,0.117501,0.367520,0.157282,0.052829,0.045453,"1,04",3.636110,0.083846,WHL,ConsumoDiscrecional
3118,2022-06-30,0.066537,0.103902,0.352467,0.097559,0.355329,0.160690,0.056837,0.051957,"1,12",2.707431,0.074095,WHL,ConsumoDiscrecional
3119,2023-06-30,0.106478,0.165522,0.426997,0.136774,0.371212,0.145186,0.070213,0.054343,"1,02",1.421566,0.036850,WHL,ConsumoDiscrecional


In [187]:
new_discretional_tickers = {}

for key in discretional_tickers.keys():
    # print(key.split("_")[1])
    new_discretional_tickers[key.split("_")[1]] = discretional_tickers[key]
    


In [190]:
df["market_ticker"] = df["ticker"].apply(lambda ticker_name: new_discretional_tickers[ticker_name])

In [195]:
'000559.SZ_CNY.csv'.split("_")

['000559.SZ', 'CNY.csv']

In [203]:
ticker = new_discretional_tickers["BRG"]
historical_price_company_dir="data/01_raw/01.4 - DatosHistoricosPrecios/Empresas/"
hist_df =None
for file in os.listdir(historical_price_company_dir):
    if file.split("_")[0] == ticker:
        hist_df = pd.read_csv(f"{historical_price_company_dir}{file}", header=[0, 1, 2])
        break

In [258]:
df["id"] = df.id.apply(lambda x: pd.to_datetime(x, utc=True))
df["year"] = df.id.apply(lambda x: x.year)

In [262]:
df["id"] = df.id.apply(lambda x: pd.to_datetime(x, utc=True))
df["year"] = df.id.apply(lambda x: x.year)

C = hist_df["Close"].transpose().reset_index().transpose().iloc[2:]
C.rename(columns={0: "Close"}, inplace=True)
C["Date"] = hist_df["Price"]["Ticker"]["Date"]
C["Date"] = C.Date.apply(lambda x: pd.to_datetime(x, utc=True))
C["year"] = C.Date.apply(lambda x: x.year)

C = C.set_index("Date").join(df[df.ticker=="BRG"].set_index("year"), on="year", how="left", lsuffix="_", rsuffix="")
C 

,Close,year,id,Return on Assets %,Return on Invested Capital %,Return On Equity %,Normalized ROIC %,Gross Profit Margin %,EBITDA Margin %,Net Income Margin %,Normalized Net Income Margin %,Current Ratio,Total Debt / Equity,% Free Cash Flow Margins,ticker,sector,market_ticker
Date,,,,,,,,,,,,,,,,,
1999-02-25 00:00:00+00:00,0.307016,1999,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1999-02-26 00:00:00+00:00,0.307016,1999,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1999-03-01 00:00:00+00:00,0.307016,1999,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1999-03-02 00:00:00+00:00,0.307016,1999,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1999-03-03 00:00:00+00:00,0.307016,1999,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-08-21 00:00:00+00:00,37.0,2025,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-08-22 00:00:00+00:00,35.130001,2025,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-08-25 00:00:00+00:00,34.59,2025,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [263]:
C[C.ticker.notna()]

,Close,year,id,Return on Assets %,Return on Invested Capital %,Return On Equity %,Normalized ROIC %,Gross Profit Margin %,EBITDA Margin %,Net Income Margin %,Normalized Net Income Margin %,Current Ratio,Total Debt / Equity,% Free Cash Flow Margins,ticker,sector,market_ticker
Date,,,,,,,,,,,,,,,,,
2004-01-01 00:00:00+00:00,1.150652,2004,2004-06-30 00:00:00+00:00,0.052282,0.013959,0.074506,0.015116,0.408122,0.097632,0.024784,0.052338,"3,06",0.443614,0.029603,BRG,ConsumoDiscrecional,BRG.AX
2004-01-02 00:00:00+00:00,1.141868,2004,2004-06-30 00:00:00+00:00,0.052282,0.013959,0.074506,0.015116,0.408122,0.097632,0.024784,0.052338,"3,06",0.443614,0.029603,BRG,ConsumoDiscrecional,BRG.AX
2004-01-05 00:00:00+00:00,1.124301,2004,2004-06-30 00:00:00+00:00,0.052282,0.013959,0.074506,0.015116,0.408122,0.097632,0.024784,0.052338,"3,06",0.443614,0.029603,BRG,ConsumoDiscrecional,BRG.AX
2004-01-06 00:00:00+00:00,1.141868,2004,2004-06-30 00:00:00+00:00,0.052282,0.013959,0.074506,0.015116,0.408122,0.097632,0.024784,0.052338,"3,06",0.443614,0.029603,BRG,ConsumoDiscrecional,BRG.AX
2004-01-07 00:00:00+00:00,1.124301,2004,2004-06-30 00:00:00+00:00,0.052282,0.013959,0.074506,0.015116,0.408122,0.097632,0.024784,0.052338,"3,06",0.443614,0.029603,BRG,ConsumoDiscrecional,BRG.AX
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-23 00:00:00+00:00,35.838661,2024,2024-06-30 00:00:00+00:00,0.087268,0.126162,0.146497,0.127855,0.364013,0.159085,0.077456,0.077456,"2,26",0.183896,0.157188,BRG,ConsumoDiscrecional,BRG.AX
2024-12-24 00:00:00+00:00,35.321568,2024,2024-06-30 00:00:00+00:00,0.087268,0.126162,0.146497,0.127855,0.364013,0.159085,0.077456,0.077456,"2,26",0.183896,0.157188,BRG,ConsumoDiscrecional,BRG.AX
2024-12-27 00:00:00+00:00,36.017654,2024,2024-06-30 00:00:00+00:00,0.087268,0.126162,0.146497,0.127855,0.364013,0.159085,0.077456,0.077456,"2,26",0.183896,0.157188,BRG,ConsumoDiscrecional,BRG.AX


In [265]:
df.ticker.nunique()

159

In [320]:
len(set(d.ticker.unique()))

159

In [323]:
len(set(d.ticker.unique()).symmetric_difference(set(new_discretional_tickers.keys()))) + 159

226